
# 25 — Direct Preference Optimization

## 01. Deriving DPO from KL-Regularized RLHF

### Motivation

RLHF traditionally learns a reward model and optimizes
a policy using reinforcement learning.

DPO directly optimizes the policy from pairwise preference
data, using a fixed reference policy.

From KL-regularized reward maximization:

$$
\pi^*(y|x)
=
\frac{1}{Z(x)}
\pi_{\mathrm{ref}}(y|x)e^{r(x,y)/\beta}
$$

We recover the reward representation:

$$
r(x,y)
=
\beta\log
\frac{\pi^*(y|x)}{\pi_{\mathrm{ref}}(y|x)}
+
\beta\log Z(x)
$$

The prompt-dependent normalization constant cancels in
the Bradley-Terry reward difference.

Replacing the optimal policy with our trainable policy
gives the DPO objective:

$$
\mathcal L_{\mathrm{DPO}}
=
-\mathbb E_{(x,y_w,y_l)}
\left[
\log\sigma\left(
\beta
\left[
\log\frac{\pi_\theta(y_w|x)}{\pi_{\mathrm{ref}}(y_w|x)}
-
\log\frac{\pi_\theta(y_l|x)}{\pi_{\mathrm{ref}}(y_l|x)}
\right]
\right)
\right]
$$

The objective uses complete-response log-probabilities,
not isolated token probabilities.


In [ ]:
import torch
import torch.nn.functional as F

beta = 0.5

# (Y=2,): reference probabilities [chosen, rejected]
reference_probs = torch.tensor([0.05, 0.95], dtype=torch.float64)

# (Y=2,): current policy probabilities [chosen, rejected]
policy_probs = torch.tensor([0.20, 0.80], dtype=torch.float64)

# (Y=2,): log-probability improvement relative to reference
log_ratios = policy_probs.log() - reference_probs.log()

# (): chosen-minus-rejected relative improvement
relative_margin = log_ratios[0] - log_ratios[1]

# (): Bradley-Terry logit under the DPO reward parameterization
dpo_logit = beta * relative_margin

# (): DPO negative log-likelihood
dpo_loss = -F.logsigmoid(dpo_logit)

# (): implicit preference probability
preference_probability = torch.sigmoid(dpo_logit)

print("Relative margin:", relative_margin.item())
print("DPO preference probability:", preference_probability.item())
print("DPO loss:", dpo_loss.item())

# A key distinction: chosen can remain less likely in the policy,
# while its improvement relative to reference is greater.
assert policy_probs[0] < policy_probs[1]
assert relative_margin > 0

print("Relative preference improvement verified.")


Relative margin: 1.5581446180465497
DPO preference probability: 0.6854801408612435
DPO loss: 0.3776357506966529
Relative preference improvement verified.



### Takeaways

DPO starts from the optimal policy of KL-regularized
reward maximization.

The reward can be represented through a policy-to-reference
log-probability ratio, up to a prompt-dependent constant.

The constant cancels when comparing two responses to the
same prompt using the Bradley-Terry model.

DPO trains a policy directly from chosen/rejected preferences.

It does not require a separately trained reward model,
a trainable Critic, or PPO rollout collection.

The reference policy remains fixed.

DPO uses relative improvements in complete-response
log-probabilities, not merely raw response likelihoods.

Next: compute DPO gradients and apply the objective to
our actual GPT policies.
